In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 99 - Reset environment (DESTRUCTIVE)
# MAGIC
# MAGIC Drops the tables, schemas, volume contents and catalogs that
# MAGIC `conf/framework.<env>.yml` names. Nothing else is reachable — the scope is derived
# MAGIC from the config, so this cannot touch a namespace the framework does not own.
# MAGIC
# MAGIC **With the default `reset_scope=none` it drops nothing and just prints an
# MAGIC inventory** of what currently exists, with row counts. Useful on its own.
# MAGIC
# MAGIC To actually drop, both widgets must be set: the scope, *and* the catalog name typed
# MAGIC out in full.
# MAGIC
# MAGIC | `reset_scope` | Tables & views | Checkpoints + landing files | Schemas & catalog |
# MAGIC |---|---|---|---|
# MAGIC | `none` | — | — | — |
# MAGIC | `tables` | dropped | **kept** | kept |
# MAGIC | `tables_and_state` | dropped | cleared | kept |
# MAGIC | `everything` | dropped | cleared | dropped (`CASCADE`) |
# MAGIC
# MAGIC **`tables` alone will look broken.** Dropping a bronze table without clearing its
# MAGIC Auto Loader checkpoint leaves the checkpoint claiming every landing file is already
# MAGIC consumed, so the next run ingests zero rows into an empty table. Use
# MAGIC `tables_and_state` for a genuine start-over.
# MAGIC
# MAGIC This notebook never touches your Git folder, `conf/` or `notebooks/`.

# COMMAND ----------


In [0]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from framework.config import FrameworkConfig  # noqa: E402
from framework.logging_utils import FrameworkLogger  # noqa: E402

# COMMAND ----------

dbutils.widgets.text("environment", "free", "Environment")
dbutils.widgets.dropdown(
    "reset_scope",
    "none",
    ["none", "tables", "tables_and_state", "everything"],
    "DESTRUCTIVE reset scope",
)
dbutils.widgets.text("reset_confirm", "", "Type the catalog name to confirm")

environment = dbutils.widgets.get("environment").strip()
reset_scope = dbutils.widgets.get("reset_scope").strip()
reset_confirm = dbutils.widgets.get("reset_confirm").strip()

cfg = FrameworkConfig.load(environment=environment)
log = FrameworkLogger(
    {"notebook": "99_reset_environment", "environment": cfg.environment}, cfg.log_level
)
free = cfg.free_edition


In [0]:

# Never drop a catalog the framework did not create. `workspace` is the documented
# fallback when CREATE CATALOG is refused, so it is reachable through a perfectly
# ordinary config - and dropping it would take the whole workspace's data with it.
# Add your own names here if your catalog holds anything besides this framework.
PROTECTED_CATALOGS = {
    "workspace", "main", "hive_metastore", "samples", "system", "__databricks_internal",
}

# COMMAND ----------

# MAGIC %md
# MAGIC ## What this config owns
# MAGIC
# MAGIC Derived from the YAML, so the blast radius is exactly the framework's namespaces.

volumes_catalog = free.get("volumes_catalog") or cfg.framework_catalog
volumes_schema = free.get("volumes_schema", "etl_volumes")
landing_volume = free.get("landing_volume", "landing")
checkpoint_volume = free.get("checkpoint_volume", "checkpoints")

raw_layer_schemas = free.get("layer_schemas", {})
if isinstance(raw_layer_schemas, list):
    raw_layer_schemas = {"__framework__": raw_layer_schemas}

In [0]:


# catalog -> schemas the framework owns in it
owned: dict = {}
owned.setdefault(cfg.framework_catalog, set()).update({cfg.control_schema, cfg.audit_schema})
owned.setdefault(volumes_catalog, set()).add(volumes_schema)
for layer, schemas in dict(raw_layer_schemas).items():
    catalog = cfg.framework_catalog if layer == "__framework__" else cfg.resolve_catalog(layer)
    owned.setdefault(catalog, set()).update(schemas or [])

volume_roots = {
    "landing": f"/Volumes/{volumes_catalog}/{volumes_schema}/{landing_volume}",
    "checkpoints": f"/Volumes/{volumes_catalog}/{volumes_schema}/{checkpoint_volume}",
}

print(f"environment  {cfg.environment}")
print(f"catalog(s)   {sorted(owned)}")
for catalog in sorted(owned):
    print(f"   {catalog}: {sorted(owned[catalog])}")
print(f"volumes      {volume_roots}")


In [0]:

# COMMAND ----------

# MAGIC %md
# MAGIC ## Inventory
# MAGIC
# MAGIC Always runs, whatever the scope. Row counts are cheap here because a Free Edition
# MAGIC test environment holds thousands of rows, not billions.


def relations_in(catalog: str, schema: str):
    """(name, is_view) for everything in a schema, or [] if the schema is absent.

    Views are listed too - the audit schema ships two - and they need DROP VIEW,
    not DROP TABLE. listTables takes an unquoted namespace, so no backticks here.
    """
    try:
        return [
            (t.name, (t.tableType or "").upper() == "VIEW")
            for t in spark.catalog.listTables(f"{catalog}.{schema}")
        ]
    except Exception:
        return []


def row_count(catalog: str, schema: str, name: str):
    try:
        return spark.sql(f"SELECT count(*) AS c FROM `{catalog}`.`{schema}`.`{name}`").collect()[0]["c"]
    except Exception:
        return None


In [0]:


inventory = []
for catalog in sorted(owned):
    for schema in sorted(owned[catalog]):
        relations = relations_in(catalog, schema)
        if not relations:
            print(f"\n{catalog}.{schema}   (empty or absent)")
            continue
        print(f"\n{catalog}.{schema}   ({len(relations)} object(s))")
        for name, is_view in sorted(relations):
            count = "" if is_view else row_count(catalog, schema, name)
            shown = "" if count is None or count == "" else f"{count:>12,} rows"
            print(f"    {'VIEW ' if is_view else 'TABLE'}  {name:<34} {shown}")
            inventory.append((catalog, schema, name, is_view))

print("\nvolume contents:")
for name, path in volume_roots.items():
    try:
        children = dbutils.fs.ls(path)
        print(f"    {name:<12} {len(children):>3} entry/entries   {path}")
        for child in children:
            print(f"        {child.name}")
    except Exception:
        print(f"    {name:<12} absent           {path}")

print(f"\n{len(inventory)} relation(s) in scope")

In [0]:


# COMMAND ----------

# MAGIC %md
# MAGIC ## Execute
# MAGIC
# MAGIC Gated twice. `reset_scope=none` stops here.

if reset_scope == "none":
    log.info("inventory only - nothing dropped", relations=len(inventory))
    print("reset_scope=none - nothing dropped.")
    print(f"To drop, set reset_scope and reset_confirm to: {cfg.framework_catalog}")
    dbutils.notebook.exit(f"INVENTORY relations={len(inventory)} catalog={cfg.framework_catalog}")

if reset_confirm != cfg.framework_catalog:
    raise ValueError(
        f"reset_scope={reset_scope} requires reset_confirm to be exactly "
        f"{cfg.framework_catalog!r} (got {reset_confirm!r}). Nothing was dropped."
    )

dropped = {"views": 0, "tables": 0, "schemas": 0, "catalogs": 0, "paths": 0}

In [0]:

# COMMAND ----------

# 1. Relations. Views first, so a view is never left dangling over a dropped table.
for catalog, schema, name, is_view in sorted(inventory, key=lambda r: not r[3]):
    kind = "VIEW" if is_view else "TABLE"
    spark.sql(f"DROP {kind} IF EXISTS `{catalog}`.`{schema}`.`{name}`")
    dropped["views" if is_view else "tables"] += 1
    log.info("dropped", kind=kind, name=f"{catalog}.{schema}.{name}")

print(f"dropped {dropped['tables']} table(s), {dropped['views']} view(s)")

# COMMAND ----------

# 2. Volume contents. The step that is easy to forget and expensive to skip - see the
#    header note about checkpoints.
if reset_scope in ("tables_and_state", "everything"):
    for name, path in volume_roots.items():
        try:
            for child in dbutils.fs.ls(path):
                dbutils.fs.rm(child.path, recurse=True)
                dropped["paths"] += 1
            log.info("volume cleared", volume=name, path=path)
            print(f"cleared {name}: {path}")
        except Exception as exc:
            log.info("volume absent", path=path, detail=str(exc)[:120])
            print(f"skipped {name} (absent): {path}")
else:
    print("checkpoints and landing files KEPT - re-running bronze will ingest nothing")

In [0]:


# COMMAND ----------

# 3. Namespaces.
if reset_scope == "everything":
    for catalog in sorted(owned):
        if catalog in PROTECTED_CATALOGS:
            # Take the framework's own schemas, leave the shared catalog alone.
            for schema in sorted(owned[catalog]):
                spark.sql(f"DROP SCHEMA IF EXISTS `{catalog}`.`{schema}` CASCADE")
                dropped["schemas"] += 1
                print(f"dropped schema {catalog}.{schema}")
            log.warning(
                "catalog is protected - dropped its framework schemas only, not the catalog",
                catalog=catalog,
            )
            continue
        # CASCADE takes the schemas, tables and managed volumes with it.
        spark.sql(f"DROP CATALOG IF EXISTS `{catalog}` CASCADE")
        dropped["catalogs"] += 1
        log.info("dropped catalog", catalog=catalog)
        print(f"dropped catalog {catalog} CASCADE")

In [0]:




# COMMAND ----------

# MAGIC %md
# MAGIC ## What remains

for catalog in sorted(owned):
    still_there = catalog in {row[0] for row in spark.sql("SHOW CATALOGS").collect()}
    if not still_there:
        print(f"{catalog}   GONE")
        continue
    for schema in sorted(owned[catalog]):
        remaining = relations_in(catalog, schema)
        print(f"{catalog}.{schema}   {len(remaining)} object(s) remaining")

log.info("reset complete", scope=reset_scope, **dropped)
print(f"\nreset complete: {dropped}")

dbutils.notebook.exit(
    f"RESET scope={reset_scope} tables={dropped['tables']} views={dropped['views']} "
    f"schemas={dropped['schemas']} catalogs={dropped['catalogs']} paths={dropped['paths']}"
)